## 야후 파이낸스(Yahoo Finance)

pip install yfinance

In [ ]:
import yfinance as yf

# Apple 주식 데이터 다운로드
# 주식 티커 심벌
aapl = yf.Ticker("AAPL")

# 한 해 동안의 주가 데이터 가져오기
hist = aapl.history(period="1y")

# 최근 5일간의 주가 데이터 출력
print(hist.tail())

                                 Open        High         Low       Close  \
Date                                                                        
2025-12-11 00:00:00-05:00  279.100006  279.589996  273.809998  278.029999   
2025-12-12 00:00:00-05:00  277.899994  279.220001  276.820007  278.279999   
2025-12-15 00:00:00-05:00  280.149994  280.149994  272.839996  274.109985   
2025-12-16 00:00:00-05:00  272.820007  275.500000  271.790009  274.609985   
2025-12-17 00:00:00-05:00  275.010010  276.160004  271.640015  271.839996   

                             Volume  Dividends  Stock Splits  
Date                                                          
2025-12-11 00:00:00-05:00  33248000        0.0           0.0  
2025-12-12 00:00:00-05:00  39532900        0.0           0.0  
2025-12-15 00:00:00-05:00  50409100        0.0           0.0  
2025-12-16 00:00:00-05:00  37648600        0.0           0.0  
2025-12-17 00:00:00-05:00  50100600        0.0           0.0  


pip install python-dotenv

In [4]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [11]:
question = "테슬라 회사의 주식 티커 심볼은 무엇인가요?"

In [12]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=question,
)

print(response.text)

테슬라(Tesla) 회사의 주식 티커 심볼은 **TSLA** 입니다.


In [13]:
question = "테슬라 회사의 주식 티커 심볼은 무엇인가요? 주식 티커 심볼만 알려주세요. 다른 말은 하지 마세요."

In [ ]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=question,
)

response.text

TSLA


In [ ]:
import yfinance as yf

aapl = yf.Ticker(response.text)

hist = aapl.history(period="1y")

hist.tail()

                                 Open        High         Low       Close  \
Date                                                                        
2025-12-11 00:00:00-05:00  448.950012  449.269989  440.329987  446.890015   
2025-12-12 00:00:00-05:00  448.089996  463.010010  441.670013  458.959991   
2025-12-15 00:00:00-05:00  469.440002  481.769989  467.660004  475.309998   
2025-12-16 00:00:00-05:00  472.209991  491.500000  465.829987  489.880005   
2025-12-17 00:00:00-05:00  488.220001  495.279999  466.200012  467.260010   

                              Volume  Dividends  Stock Splits  
Date                                                           
2025-12-11 00:00:00-05:00   55979500        0.0           0.0  
2025-12-12 00:00:00-05:00   95656700        0.0           0.0  
2025-12-15 00:00:00-05:00  114542200        0.0           0.0  
2025-12-16 00:00:00-05:00  107608100        0.0           0.0  
2025-12-17 00:00:00-05:00  106163800        0.0           0.0  


### 주가 분석기

In [17]:
stock_item = input("주식 종목: ").strip()
stock_item

'테슬라'

In [18]:
question = f"{stock_item} 회사의 주식 티커 심볼은 무엇인가요? 주식 티커 심볼만 알려주세요. 다른 말은 하지 마세요."
question

'테슬라 회사의 주식 티커 심볼은 무엇인가요? 주식 티커 심볼만 알려주세요. 다른 말은 하지 마세요.'

In [19]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=question,
)

response.text

'TSLA'

In [22]:
import yfinance as yf
from datetime import datetime

def get_stock_data(ticker_symbol):
    print("\n[조회 방식 선택]")
    print("1. 상대 기간 (1d, 5d, 1mo, 1y, max 등)")
    print("2. 절대 기간 (시작일과 종료일 직접 입력)")
    
    choice = input("\n방식을 선택하세요 (1 또는 2): ")

    try:
        stock = yf.Ticker(ticker_symbol)
        
        if choice == '1':
            period = input("조회 기간을 입력하세요 (예: 1mo): ").lower()
            data = stock.history(period=period)
            
        elif choice == '2':
            start_input = input("시작일을 입력하세요 (YYYY-MM-DD): ")
            end_input = input("종료일을 입력하세요 (YYYY-MM-DD): ")
            
            # 형식 검증
            datetime.strptime(start_input, '%Y-%m-%d')
            datetime.strptime(end_input, '%Y-%m-%d')
            
            data = stock.history(start=start_input, end=end_input)
            
        else:
            print("잘못된 선택입니다.")
            return None

        # 데이터 검증 및 튜플 생성
        if data.empty:
            print(f"\n데이터를 불러오지 못했습니다. 종목 코드나 기간을 확인하세요.")
            return None
        else:
            # 1. 실제 데이터 상의 시작일과 종료일 추출 (문자열 형식)
            # 인덱스가 DatetimeIndex이므로 날짜 부분만 추출
            actual_start_date = data.index[0].strftime('%Y-%m-%d')
            actual_end_date = data.index[-1].strftime('%Y-%m-%d')
            
            # 2. LLM 전달용 텍스트 테이블 변환
            data_table = data.to_string() # 혹은 data.to_markdown() 사용 권장
            
            return (actual_start_date, actual_end_date, data_table)
            
    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")
        return None

# --- 실행 및 적용 ---
# 기존 코드의 ticker_symbol 파라미터 전달 (예: 'TSLA')
ticker = "TSLA"  
result = get_stock_data(ticker)

if result:
    start_date, end_date, data_table = result
    
    # 이전에 작성한 f-string 함수와 연동 가능
    # final_prompt = generate_stock_analysis_prompt(start_date, end_date, data_table)
    # print(final_prompt)
    print(f"\n[성공] {start_date}부터 {end_date}까지의 데이터를 추출했습니다.")


[조회 방식 선택]
1. 상대 기간 (1d, 5d, 1mo, 1y, max 등)
2. 절대 기간 (시작일과 종료일 직접 입력)

[성공] 2025-11-18부터 2025-12-17까지의 데이터를 추출했습니다.


In [23]:
prompt = f"""
# Role
너는 데이터에 근거하여 리스크를 관리하고 수익 기회를 포착하는 냉철한 자산운용가이다.

# Task
제공된 주가 데이터를 바탕으로 아래 지정된 분석 기간에 대한 '투자 전략 보고서'를 작성하라.

# Analysis Period
- 시작일: {start_date}
- 종료일: {end_date}

# Instructions
1. **차트 패턴 및 지지/저항 분석**: 
   - 해당 기간 내에서 반복적으로 반등이 일어난 '지지선(Support)'과 돌파에 실패한 '저항선(Resistance)'을 수치로 도출하라.
2. **모멘텀 및 과열 진단**: 
   - {start_date} 대비 {end_date}의 가격 변동률을 계산하고, 상승/하락의 강도가 거래량에 의해 지지되고 있는지 분석하라.
   - 특히 거래량이 급증한 지점을 찾아, 그것이 '매수 가열'인지 '패닉 셀'인지 정의하라.
3. **리스크 평가 및 변동성 분석**: 
   - {end_date} 기준, 최근 3거래일간의 종가 흐름을 분석하여 하락 전환의 신호가 있는지 검토하라.
   - 일일 변동성(High-Low)이 가장 컸던 날을 특정하고, 그날의 시가 대비 종가 이격도가 향후 방향성에 주는 함의를 기술하라.
4. **최종 Action Plan**: 
   - 현재 시점에서 신규 진입, 보유 유지(Hold), 혹은 비중 축소(Reduce) 중 하나의 의견을 제시하고 그 근거를 냉정하게 서술하라.

# Data
{data_table}
""".strip()

In [24]:
prompt

"# Role\n너는 데이터에 근거하여 리스크를 관리하고 수익 기회를 포착하는 냉철한 자산운용가이다.\n\n# Task\n제공된 주가 데이터를 바탕으로 아래 지정된 분석 기간에 대한 '투자 전략 보고서'를 작성하라.\n\n# Analysis Period\n- 시작일: 2025-11-18\n- 종료일: 2025-12-17\n\n# Instructions\n1. **차트 패턴 및 지지/저항 분석**: \n   - 해당 기간 내에서 반복적으로 반등이 일어난 '지지선(Support)'과 돌파에 실패한 '저항선(Resistance)'을 수치로 도출하라.\n2. **모멘텀 및 과열 진단**: \n   - 2025-11-18 대비 2025-12-17의 가격 변동률을 계산하고, 상승/하락의 강도가 거래량에 의해 지지되고 있는지 분석하라.\n   - 특히 거래량이 급증한 지점을 찾아, 그것이 '매수 가열'인지 '패닉 셀'인지 정의하라.\n3. **리스크 평가 및 변동성 분석**: \n   - 2025-12-17 기준, 최근 3거래일간의 종가 흐름을 분석하여 하락 전환의 신호가 있는지 검토하라.\n   - 일일 변동성(High-Low)이 가장 컸던 날을 특정하고, 그날의 시가 대비 종가 이격도가 향후 방향성에 주는 함의를 기술하라.\n4. **최종 Action Plan**: \n   - 현재 시점에서 신규 진입, 보유 유지(Hold), 혹은 비중 축소(Reduce) 중 하나의 의견을 제시하고 그 근거를 냉정하게 서술하라.\n\n# Data\n                                 Open        High         Low       Close     Volume  Dividends  Stock Splits\nDate                                                                                                         \n2025-11-18 00:00:00-05:00 

In [26]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

response.text

"## 투자 전략 보고서\n\n**작성일:** 2025-12-17 (데이터 분석 기준)\n**종목:** [데이터에 명시된 가상의 종목]\n**분석 기간:** 2025-11-18 ~ 2025-12-17\n\n---\n\n### 1. 차트 패턴 및 지지/저항 분석\n\n제공된 기간 내 주가 흐름을 분석한 결과, 다음과 같은 지지 및 저항 수준이 도출됩니다.\n\n*   **주요 지지선(Support Levels):**\n    *   **1차 지지선: 420.00 - 425.00**\n        *   2025-12-01 (Low 425.29), 2025-12-02 (Low 422.12)에서 해당 구간에서 반등 시도가 관찰되었습니다.\n    *   **2차 지지선: 435.00 - 440.00**\n        *   2025-12-08 (Low 435.25), 2025-12-09 (Low 435.70)에서 하락세가 멈추고 반등하는 모습을 보였습니다.\n        *   현재 종가(467.26)를 고려할 때, 이 구간은 향후 주요 지지선으로 작용할 가능성이 높습니다.\n    *   **장기 지지선: 390.00 - 395.00**\n        *   2025-11-20 (Low 394.74), 2025-11-21 (Low 383.76)에 단기 저점을 형성한 후 반등했습니다. 현재 주가와는 다소 거리가 있으나, 큰 폭의 조정 시 참고할 수 있는 수준입니다.\n\n*   **주요 저항선(Resistance Levels):**\n    *   **1차 저항선: 455.00 - 460.00**\n        *   2025-12-05 (High 458.87), 2025-12-10 (High 456.88)에서 돌파 시도가 있었으나 일시적으로 저항에 부딪히는 모습이 관찰되었습니다.\n    *   **2차 저항선: 490.00 - 495.00**\n        *   2025-12-16 (High 491.50), 2025-12-17 (High 495.28

In [42]:
prompt = f"""
너는 데이터 시각화 및 금융 분석 전문가이다. 
제공된 주가 데이터를 정밀 분석하여, 투자자가 한눈에 시장 상황을 파악할 수 있는 '종목 분석 대시보드'를 HTML과 CSS로 작성하라.

**분석 요청 사항:**
1. **기간:** {start_date} ~ {end_date} 의 추세를 반영할 것.
2. **주요 지표:** 해당 기간의 누적 수익률, 최고가/최저가, 평균 거래량을 포함할 것.
3. **투자 전략(Action Plan):** 현재의 모멘텀을 분석하여 '매수/보유/관망' 중 하나를 시각적으로 강조하여 표시할 것.
4. **리스크 요인:** 최근 변동성을 바탕으로 주의해야 할 가격대를 리스트업할 것.
5. **전문가 분석 의견(Expert Commentary):** - 현재 데이터 흐름에서 포착되는 독특한 패턴이나 수치적 특징을 냉철하게 분석하여 기술할 것.
   - 단순 요약을 넘어, 거래량과 가격의 상관관계에 근거한 전문가적 통찰을 2~3문장으로 제시할 것.

**디자인 가이드라인:**
- **포맷:** 현대적이고 깔끔한 다크 모드 금융 대시보드 스타일.
- **제약:** <img> 태그 등 외부 이미지는 사용하지 말고, 이모지(📈, 📉, 💰, ⚠️)와 CSS 도형/레이아웃(Flexbox, Grid)만 사용할 것.
- **출력:** HTML 문서 코드 외에 어떠한 부연 설명이나 텍스트도 작성하지 마라. 오직 `<!DOCTYPE html>`로 시작하는 코드만 응답하라.

**주가 데이터:**
{data_table}
"""

In [43]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

response.text

'```html\n<!DOCTYPE html>\n<html lang="ko">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>종목 분석 대시보드</title>\n    <style>\n        @import url(\'https://fonts.googleapis.com/css2?family=Roboto:wght@300;404;700&display=swap\');\n\n        :root {\n            --bg-color: #1a1a2e;\n            --card-bg: #16213e;\n            --text-color: #e0e0e0;\n            --header-color: #0f3460;\n            --accent-color: #e94560;\n            --positive-color: #28a745;\n            --negative-color: #dc3545;\n            --hold-color: #ffc107; /* Warm yellow/orange for HOLD */\n            --border-color: #0f3460;\n            --shadow-color: rgba(0, 0, 0, 0.3);\n        }\n\n        body {\n            font-family: \'Roboto\', sans-serif;\n            margin: 0;\n            padding: 20px;\n            background-color: var(--bg-color);\n            color: var(--text-color);\n            line-height: 1.6;\n          

In [44]:
result = response.text
result = result.replace("```html","").replace("```","")
print(result)


<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>종목 분석 대시보드</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Roboto:wght@300;404;700&display=swap');

        :root {
            --bg-color: #1a1a2e;
            --card-bg: #16213e;
            --text-color: #e0e0e0;
            --header-color: #0f3460;
            --accent-color: #e94560;
            --positive-color: #28a745;
            --negative-color: #dc3545;
            --hold-color: #ffc107; /* Warm yellow/orange for HOLD */
            --border-color: #0f3460;
            --shadow-color: rgba(0, 0, 0, 0.3);
        }

        body {
            font-family: 'Roboto', sans-serif;
            margin: 0;
            padding: 20px;
            background-color: var(--bg-color);
            color: var(--text-color);
            line-height: 1.6;
            display: flex;
            justify-conte

In [47]:
formatted_time = datetime.now().strftime('%Y%m%d')
formatted_time

'20251218'

In [49]:
# html 인포그래픽 생성
file = open(f"result_analysis_{formatted_time}.html", "w", encoding="utf8")

file.write(str(result))
file.close()